# Podcast Notes, Summarization,Translation and Image Processing using Phi‑4 Multimodal LLM with NVIDIA NIM Microservices

This notebook demonstrates a complete workflow using the [**Phi‑4 Multimodal LLM**](https://azure.microsoft.com/en-us/blog/empowering-innovation-the-next-generation-of-the-phi-family/) model. Below are some key model details (from the internal model card):

- **Parameters:** 5.6B
- **Inputs:** Text, Image, Audio
- **Context Length:** 128K tokens
- **Training Data:** 5T text tokens, 2.3M speech hours, 1.1T image-text tokens
- **Supported Languages:** Multilingual text and audio (e.g. English, Chinese, German, French, etc.)

The Phi-4 LLM will be accelerated with NVIDIA NIM Microservices in this tutorial. NVIDIA NIM Microservices that is a set of easy-to-use inference microservices for accelerating the deployment of foundation models on any cloud or data center and helping to keep your data secure.

We will be using [preview NIM microservice API](https://build.nvidia.com/microsoft/phi-4-multimodal-instruct) for Phi-4 through the  [NVIDIA API Catalog](https://build.nvidia.com/microsoft)

This notebook covers:
1. A podcast notes and summarization use-case (with long-audio chunking).
2. Translation of the transcript and summary into another language.
3. Image Processing with Phi‑4 Multimodal

Ensure that you have the required dependencies (e.g. `pydub`, `requests`, `Pillow`) installed. If the virtual environment (venv) was created successfully, there’s no need to run the next cell.

In [1]:
# !pip install pydub requests Pillow 

## Audio Processing with Phi‑4 Multimodal
This section demonstrates three practical use cases using Phi‑4 for audio processing: 

1. Generate detailed notes from audio
2. Summarization
3. Translation


### Utility Functions for Audio Processing

In this section, we define helper functions to process audio files, such as converting an audio segment to a base64-encoded string 
and splitting long audio into manageable chunks for transcription.

- **audio_to_base64**: Converts audio segments to base64-encoded strings for API transmission
- **generate_notes_chunk**: Processes individual audio chunks and generates transcription
- **generate_detailed_notes**: Handles long audio files by splitting them into manageable chunks
- **refine_transcription_to_notes**: Transforms raw transcription into well-formatted notes
- **summarize_notes**: Creates a concise summary from the detailed notes
- **translate_text**: Translates content to other languages while preserving formatting
- **save_text_to_file**: Exports results to text files

In [2]:
import requests
import base64
from io import BytesIO
from pydub import AudioSegment

API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

def audio_to_base64(audio_segment):
    """Convert a pydub AudioSegment to a base64-encoded WAV string."""
    buffer = BytesIO()
    audio_segment.export(buffer, format="mp3")
    return base64.b64encode(buffer.getvalue()).decode()

def generate_notes_chunk(audio_chunk, api_key, chunk_index, total_chunks):
    """
    Generate detailed notes for a single audio chunk.
    
    The prompt instructs the model to produce detailed notes without any extra commentary.
    """
    audio_b64 = audio_to_base64(audio_chunk)
    prompt = (
        f"Transcribe the following audio accurately. "
        f"This is segment {chunk_index+1} of {total_chunks}. "
        "Please do not include any system commentary or self-referential text."
    )
    # Append the audio data (encoded in base64)
    prompt += f' <audio src="data:audio/wav;base64,{audio_b64}" />'
    
    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    payload = {
        "model": "microsoft/phi-4-multimodal-instruct",
        "messages": [
            {
                "role": "system",
                "content": "You're a transcription assistant."
            },
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.1,
        "top_p": 0.7,
        "stream": False
    }
    
    response = requests.post(API_URL, headers=headers, json=payload)
    result = response.json()
    try:
        text = result["choices"][0]["message"]["content"].strip()
    except Exception as e:
        text = f"[Error generating notes for chunk {chunk_index+1}: {e}]"
    return text

def generate_detailed_notes(audio_path, api_key, chunk_duration_ms=30000):
    """
    Process a long audio file by splitting it into chunks, generating detailed notes for each chunk,
    and concatenating the results.
    """
    # Load audio file using pydub
    audio = AudioSegment.from_file(audio_path)
    total_duration = len(audio)
    total_chunks = (total_duration // chunk_duration_ms) + (1 if total_duration % chunk_duration_ms > 0 else 0)
    
    notes_chunks = []
    for i in range(total_chunks):
        start_ms = i * chunk_duration_ms
        end_ms = min((i+1) * chunk_duration_ms, total_duration)
        chunk = audio[start_ms:end_ms]
        print(f"Processing chunk {i+1}/{total_chunks} (from {start_ms}ms to {end_ms}ms)...")
        notes = generate_notes_chunk(chunk, api_key, i, total_chunks)
        notes_chunks.append(notes)
    
    transcription = "\n\n".join(notes_chunks)
    return transcription


def refine_transcription_to_notes(transcription, api_key):
    """
    Given a raw transcription, remove any system prompt lines and generate coherent, well-formatted detailed notes.
    """
    # Remove unwanted system phrases
    cleaned = transcription.replace("You are a helpful assistant.", "").replace("you are a helpful assistant.", "")
    
    # Build a prompt instructing the LLM to produce formatted notes
    prompt = (
        "Based on the transcription below, generate well-formatted, detailed notes that capture the main points. "
        "Organize the notes using bullet points or numbered lists for key points and separate paragraphs clearly for readability. "
        "Do not include any system commentary or self-referential text.\n\n"
        "Transcription:\n"
        f"{cleaned}"
    )
    
    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    payload = {
        "model": "microsoft/phi-4-multimodal-instruct",
        "messages": [
            {"role": "system", "content": "You are a note-taking assistant. Provide only detailed, well-formatted notes."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.1,
        "top_p": 0.7,
        "stream": False
    }
    
    response = requests.post(API_URL, headers=headers, json=payload)
    result = response.json()
    try:
        notes = result["choices"][0]["message"]["content"].strip()
    except Exception as e:
        notes = f"[Error generating refined notes: {e}]"
    return notes



def summarize_notes(notes, api_key):
    """
    Generate a concise summary based on the detailed notes.
    """
    prompt = (
        "Based on the detailed notes provided below, please generate a concise summary of the content. "
        "Do not include any extra commentary or self-referential text.\n\n"
        f"{notes}"
    )
    
    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    payload = {
        "model": "microsoft/phi-4-multimodal-instruct",
        "messages": [
            {"role": "system", "content": "You are a summarization assistant. Provide only a concise summary."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 512,
        "temperature": 0.1,
        "top_p": 0.7,
        "stream": False
    }
    
    response = requests.post(API_URL, headers=headers, json=payload)
    result = response.json()
    try:
        summary = result["choices"][0]["message"]["content"].strip()
    except Exception as e:
        summary = f"[Error generating summary: {e}]"
    return summary


def translate_text(text, target_lang, api_key):
    """
    Translate the given text into the target language using the Phi‑4 API,
    preserving all bullet points, formatting, and structure.
    """
    prompt = (
        f"Translate the following text to {target_lang} exactly as it is, "
        "preserving all bullet points, formatting, and structure. Do not omit any sections.\n\n"
        f"{text}"
    )
    
    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    payload = {
        "model": "microsoft/phi-4-multimodal-instruct",
        "messages": [
            {
                "role": "system",
                "content": "You are a translation assistant. Provide only the translated text with the original formatting preserved."
            },
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1500,
        "temperature": 0.1,
        "top_p": 0.7,
        "stream": False
    }
    
    response = requests.post(API_URL, headers=headers, json=payload)
    result = response.json()
    try:
        translated_text = result["choices"][0]["message"]["content"].strip()
    except Exception as e:
        translated_text = f"[Error generating translation: {e}]"
    return translated_text


def save_text_to_file(text, filename):
    """Save the given text to a file."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Text saved to {filename}")


### Workflow for detailed notes generation


In [3]:
# Set your NVIDIA API key
API_KEY = "REMOVED_API_KEY"

In [4]:
# Set your  podcast audio file path
podcast_audio_path = "test_samples/sample_tedx.mp3"  # update with your file path

# Generate detailed notes from the audio file (with 30-second chunks by default)
print("Starting detailed note generation...")
transcription = generate_detailed_notes(podcast_audio_path, API_KEY, chunk_duration_ms=30000)
detailed_notes = refine_transcription_to_notes(transcription, API_KEY)

print("\n--- Detailed Notes ---\n")
print(detailed_notes)

# Generate a summary of the detailed notes
print("\nGenerating summary...")
summary = summarize_notes(detailed_notes, API_KEY)

print("\n--- Summary ---\n")
print(summary)


combined_text = detailed_notes + "\n\n--- SUMMARY ---\n\n" + summary

# # Save detailed notes and summary to a text file
# save_text_to_file(combined_text, "podcast_detailed_notes.txt")


Starting detailed note generation...
Processing chunk 1/12 (from 0ms to 30000ms)...
Processing chunk 2/12 (from 30000ms to 60000ms)...
Processing chunk 3/12 (from 60000ms to 90000ms)...
Processing chunk 4/12 (from 90000ms to 120000ms)...
Processing chunk 5/12 (from 120000ms to 150000ms)...
Processing chunk 6/12 (from 150000ms to 180000ms)...
Processing chunk 7/12 (from 180000ms to 210000ms)...
Processing chunk 8/12 (from 210000ms to 240000ms)...
Processing chunk 9/12 (from 240000ms to 270000ms)...
Processing chunk 10/12 (from 270000ms to 300000ms)...
Processing chunk 11/12 (from 300000ms to 330000ms)...
Processing chunk 12/12 (from 330000ms to 355544ms)...

--- Detailed Notes ---

- Speaker claims to have nothing to say but will make it seem intelligent through speaking.
- Uses hand gestures, adjusts glasses, and asks audience if they've been asked a question before.
- Reacts to audience's participation with a personal anecdote, then shifts to an intellectual point.
- Mentions the impo

### Translation

The following cell translates the combined notes and summary into another language (e.g., Spanish, Arabic).

In [5]:
target_language = "Arabic"
print(f"\nTranslating transcript and summary to {target_language}...")
translated_text = translate_text(combined_text, target_language, API_KEY)

print("\n--- Translated Text ---\n")
print(translated_text)

# # Save the translated text to a file
# save_text_to_file(translated_text, "podcast_transcription_translated.txt")


Translating transcript and summary to Arabic...

--- Translated Text ---

--- ملخص ---
يقول المحاور أنه لا لديه شيئًا للقول ولكنه سيجعل يبدو ذكيًا من خلال الكلام. يستخدم الإيماءات اليدوية، ويعدل نظاراته، ويستفسر من الجمهور إذا كان قد تم سؤالهم من قبل. يتفاعل مع المشاركة من الجمهور بملاحظة شخصية، ثم يتجه إلى نقطة عقلية. يذكر أهمية عضو الملاحظات (المترجم الصوتي) ويشارك معلومات شخصية (ارتفاع: 70.5 بوصة). يقدم سلسلة من الأرقام والرسوم البيانية، ويزعم أن هذه ذات معنى حقًا. يستخدم الغموض والعبارات غير المنطقية أثناء الإيماءات والركض، مما يشير إلى بناء نحو استنتاج. يذكر الجمهور لتذكر بداية وحل المحادثة، ويؤكد على نقص المحتوى، ويختتم بإشادة الجمهور ويقول أن شيء لم يغير. ---


## Image Processing with Phi‑4 Multimodal

Beyond audio, Phi‑4 natively understands images. This section demonstrates three practical use cases:

1. **Image Description** — generate a detailed natural-language caption
2. **Visual Q&A** — ask targeted questions about an image
3. **Text Extraction (OCR)** — read and transcribe text embedded in images

### Image Utility Functions

- **image_to_base64**: Loads any image file and encodes it as a base64 JPEG string
- **analyze_image**: Core function — sends an image + prompt to the Phi‑4 API and returns the model's response

In [6]:
from PIL import Image

def image_to_base64(image_path):
    """Load an image and convert it to a base64-encoded JPEG string."""
    with Image.open(image_path) as img:
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buffer = BytesIO()
        img.save(buffer, format="JPEG")
        return base64.b64encode(buffer.getvalue()).decode()


def analyze_image(image_path, prompt, api_key, system_message="You are a helpful visual assistant.", max_tokens=512):
    """
    Send an image with a text prompt to the Phi‑4 multimodal API.
    
    Args:
        image_path: Path to the image file (JPEG, PNG, etc.)
        prompt: The question or instruction for the model
        api_key: NVIDIA NIM API key
        system_message: System role context
        max_tokens: Maximum tokens in the response
    Returns:
        The model's text response
    """
    img_b64 = image_to_base64(image_path)

    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    payload = {
        "model": "microsoft/phi-4-multimodal-instruct",
        "messages": [
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],
        "max_tokens": max_tokens,
        "temperature": 0.1,
        "top_p": 0.7,
        "stream": False
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    result = response.json()
    try:
        return result["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"[Error analyzing image: {e}]\nResponse: {result}"

### Image Description

Generate a detailed natural-language description of any image. Useful for accessibility captions, content indexing, or as input to downstream text pipelines.

In [7]:
image_path = "test_samples/test_image.jpeg"  # update with your image path

# --- Use Case 1: Image Description ---
print("=== Use Case 1: Image Description ===\n")
description = analyze_image(
    image_path,
    prompt="Describe this image in detail. Include objects, people, setting, colors, and any notable elements.",
    api_key=API_KEY,
    system_message="You are a visual description assistant. Provide detailed, accurate image descriptions.",
    max_tokens=512
)
print(description)
# save_text_to_file(description, "image_description.txt")

=== Use Case 1: Image Description ===

The image depicts a scenic view of a river flowing through a mountainous landscape. The river is surrounded by large rocks and boulders, with clear, turquoise water rushing over them. The river is flanked by lush green trees and vegetation on both sides. The mountains in the background are covered with dense forests and have a mix of green and brown hues. The sky is partly cloudy with patches of blue visible, adding to the picturesque setting. The overall scene is vibrant and natural, showcasing the beauty of the natural environment.


### Visual Q&A

Ask the model targeted questions about a specific image. This is useful for automated image analysis, content moderation, or extracting structured insights.

In [8]:
image_path = "test_samples/test_image.jpeg"  # update with your image path

# --- Use Case 2: Visual Q&A ---
print("=== Use Case 2: Visual Q&A ===\n")
questions = [
    "What is the main subject of this image?",
    "What colors are most prominent?",
    "What mood or atmosphere does this image convey?",
]
for question in questions:
    answer = analyze_image(
        image_path,
        prompt=question,
        api_key=API_KEY,
        system_message="You are a visual Q&A assistant. Answer questions about images concisely and accurately.",
        max_tokens=256
    )
    print(f"Q: {question}")
    print(f"A: {answer}\n")

=== Use Case 2: Visual Q&A ===

Q: What is the main subject of this image?
A: river

Q: What colors are most prominent?
A: blue, green, and brown

Q: What mood or atmosphere does this image convey?
A: The image conveys a serene and natural atmosphere, with a flowing river surrounded by lush greenery and mountains in the background.



### Text Extraction (OCR)

Phi‑4 can read and transcribe text directly from images — useful for extracting content from slides, screenshots, whiteboards, or scanned documents.

In [9]:
image_path = "test_samples/testocr.png"  # update with your image path

# --- Use Case 3: Text Extraction (OCR) ---
print("=== Use Case 3: Text Extraction (OCR) ===\n")
extracted_text = analyze_image(
    image_path,
    prompt="Extract and transcribe all text visible in this image. Preserve the layout and structure as closely as possible.",
    api_key=API_KEY,
    system_message="You are an OCR assistant. Extract all text from images accurately, preserving layout.",
    max_tokens=1024
)
print(extracted_text)
# save_text_to_file(extracted_text, "image_extracted_text.txt")

=== Use Case 3: Text Extraction (OCR) ===

This is a lot of 12 point text to test the ocr code and see if it works on all types of file format.
The quick brown dog jumped over the lazy fox. The quick brown dog jumped over the lazy fox. The quick brown dog jumped over the lazy fox. The quick brown dog jumped over the lazy fox.


In [10]:
image_path = "test_samples/testocr_ar.png"  # update with your image path

# --- Use Case 3: Text Extraction (OCR) ---
print("=== Use Case 3: Text Extraction (OCR) ===\n")
extracted_text = analyze_image(
    image_path,
    prompt="Extract and transcribe all text visible in this image. Preserve the layout and structure as closely as possible.",
    api_key=API_KEY,
    system_message="You are an OCR assistant. Extract all text from images accurately, preserving layout.",
    max_tokens=1024
)
print(extracted_text)
# save_text_to_file(extracted_text, "image_extracted_text.txt")

=== Use Case 3: Text Extraction (OCR) ===

Notebook Computer: (الخطاب) أو جهاز الكمبيوتر المكتوب أو جهاز الكمبيوتر المحمول هو جهاز صغير مصمم لفك تشفير المعلومات. (Laptop) هو جهاز صغير مصمم لفك تشفير المعلومات. (LCD) هو جهاز صغير مصمم لفك تشفير المعلومات. (LED) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم لفك تشفير المعلومات. (Tablet) هو جهاز صغير مصمم ل